# smiDE Differential Expression Pipeline for CosMX RNA Data

## Instructions

- **Pre-requisite**: The Seurat object must contain continuous predictor columns
  (created in the Predictors notebook). Binning can be done here or in advance.
- Helper functions are loaded from `helper_functions.R`.
- This notebook implements the NanoString smiDE pipeline in three stages:
  1. **hastyDE** — fast screening (100+ genes, no spatial correction)
  2. **Standard smiDE** — negative binomial model with neighbor expression correction
  3. **Spatial smiDE** — same + GP spatial random effect (resource-intensive, optional)
- Each stage is toggled via `run_*` flags. Expensive steps are cached via `compute_*` flags.
- After a kernel restart, set `compute_*` flags to `FALSE` to reload from cache.
- **Celltype-specific binning**: Set `bin_within_celltype = TRUE` to create `de_var`
  by binning a continuous predictor within `celltype_oi` cells (recommended for power).
  Set to `FALSE` if `de_var` already exists as a categorical column.
- Reference: https://nanostring-biostats.github.io/CosMx-Analysis-Scratch-Space/posts/vignette-basic-analysis-updated/03_DE.html

## Imports

In [2]:
# Load Packages
library(Seurat)
library(dplyr)
library(ggplot2)
library(gridExtra)
library(patchwork)
library(data.table)
library(Matrix)
library(matrixStats)
library(ggrepel)
library(pheatmap)
library(scales)
library(smiDE)

# Load helper functions
source("helper_functions.R")

## Configuration

All parameters for this run are defined in the cell below.
Change values here, then re-run the notebook.

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell only
# ============================================================

# --- Study & Paths ---
study_name    <- "TMA18"
seu_file_path <- file.path("../outputs", study_name, "seurat_objects",
                           "annotated_object_TMA18_louvain_final_hieratype.RDS")
out_dir       <- "../outputs"
ASSAY_NAME    <- "RNA"

# --- Column Names ---
sdimx_col     <- "x_slide_mm"
sdimy_col     <- "y_slide_mm"
celltype_col  <- "celltype"          # cell type annotation column
cellid_col    <- "cell_ID_new"

# --- DE Parameters ---
celltype_oi   <- "cd8_exhausted"     # cell type of interest
de_var        <- "neighboring_tumor_bins_ct"  # metadata column to test (created below if bin_within_celltype = TRUE)

# --- Celltype-Specific Binning ---
# Bins a continuous predictor within celltype_oi cells only,
# creating de_var with balanced groups for that cell type.
# Set to FALSE if de_var already exists in metadata.
bin_within_celltype <- TRUE
bin_source_col      <- "neighboring_tumor_count"  # continuous column from Predictors notebook
bin_n_bins          <- 3
bin_labels          <- c("Few neighbors", "Some neighbors", "Many neighbors")  # low -> high

# --- Overlap Ratio ---
radius                     <- 0.05
split_neighbors_by_colname <- "region"
overlap_threshold          <- 1      # genes with ratio > threshold are excluded

# --- smiDE Model ---
use_random_intercept <- TRUE
random_intercept_col <- "study_id"
family               <- "nbinom2"

# --- Pipeline Toggles ---
run_hasty    <- TRUE    # Stage 1: fast exploratory screen
run_standard <- TRUE    # Stage 2: full smiDE (no spatial)
run_spatial  <- FALSE   # Stage 3: spatial smiDE (resource-intensive)

# --- Spatial DE Settings ---
spatial_method      <- "GP_INLA"   # "GP_INLA" or "GP_Matern"
spatial_top_n       <- 10          # number of genes for spatial DE
spatial_source      <- "hasty"     # rank genes from: "hasty" or "standard"
spatial_num_threads <- 4
spatial_k           <- 10          # k-means clusters for GP_Matern

# --- Results & Visualization ---
fdr_cut     <- 0.05
x_cut       <- 0.5   # log2FC threshold for volcano plots
top_n_viz   <- 10    # top genes to show in tables / plots
bin_order   <- NULL  # set e.g. c("Few neighbors", "Some neighbors", "Many neighbors") or NULL

genes_of_interest <- c(   # highlighted on volcano plots
    "BATF", "CASP3", "CCL3", "CD160", "CD244", "CD86",
    "CTLA4", "EOMES", "FASLG", "HAVCR2", "IER5", "IRF4",
    "KLRG1", "LAG3", "LILRB4", "MDFIC", "PBX3", "PDCD1",
    "PLSCR1", "PTGER2", "RGS16", "TIGIT", "TNFRSF9", "TNFSF9"
)

# --- Cache Flags (flip to FALSE to reload from disk) ---
compute_overlapres <- TRUE
compute_pre_de     <- TRUE
compute_hasty      <- TRUE
compute_standard   <- TRUE
compute_spatial    <- TRUE

# ============================================================
# DERIVED — do not edit below this line
# ============================================================
run_tag     <- paste0(study_name, "_", celltype_oi, "_", de_var)
smide_dir   <- file.path(out_dir, study_name, "smiDE")
run_out_dir <- file.path(smide_dir, run_tag)
cache_dir   <- file.path(run_out_dir, "cache")
results_dir <- file.path(run_out_dir, "results")
figures_dir <- file.path(run_out_dir, "figures")

dir.create(cache_dir,   showWarnings = FALSE, recursive = TRUE)
dir.create(results_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(figures_dir, showWarnings = FALSE, recursive = TRUE)

cat("Run tag:    ", run_tag, "\n")
cat("Output dir: ", run_out_dir, "\n")

In [ ]:
# Snapshot of all config parameters for reproducibility
config <- list(
    study_name = study_name, ASSAY_NAME = ASSAY_NAME,
    celltype_oi = celltype_oi, de_var = de_var,
    celltype_col = celltype_col, cellid_col = cellid_col,
    sdimx_col = sdimx_col, sdimy_col = sdimy_col,
    bin_within_celltype = bin_within_celltype,
    bin_source_col = bin_source_col, bin_n_bins = bin_n_bins,
    radius = radius, split_neighbors_by_colname = split_neighbors_by_colname,
    overlap_threshold = overlap_threshold,
    use_random_intercept = use_random_intercept,
    random_intercept_col = random_intercept_col, family = family,
    run_hasty = run_hasty, run_standard = run_standard, run_spatial = run_spatial,
    spatial_method = spatial_method, spatial_top_n = spatial_top_n,
    fdr_cut = fdr_cut, x_cut = x_cut,
    run_tag = run_tag, timestamp = Sys.time()
)
saveRDS(config, file.path(run_out_dir, paste0("run_config_", run_tag, ".RDS")))
message("Config saved.")

## Data Loading & Preparation

In [ ]:
# Load Seurat object
seu <- readRDS(seu_file_path)
DefaultAssay(seu) <- ASSAY_NAME
cat("Loaded:", ncol(seu), "cells x", nrow(seu), "genes\n")

In [ ]:
# Extract counts: cells in rows, genes in columns
counts     <- Matrix::t(seu[[ASSAY_NAME]]$counts)
neg_counts <- Matrix::t(seu[["negprobes"]]$counts)
metadata   <- seu@meta.data
metadata$neg_total_counts <- Matrix::rowSums(neg_counts)
metadata$negmean          <- Matrix::rowMeans(neg_counts)
xy <- as.matrix(metadata[, c(sdimx_col, sdimy_col)])

# Alignment check
stopifnot(all(rownames(counts) == rownames(metadata)))
stopifnot(all(rownames(counts) == rownames(xy)))

# Project-standard sparse normalization
scale_row <- mean(metadata$nCount_RNA) / metadata$nCount_RNA
norm <- counts
norm@x <- norm@x * scale_row[norm@i + 1L]

# Store normalized data back in Seurat for visualization helpers
seu <- Seurat::SetAssayData(seu, layer = "data", new.data = Matrix::t(norm))

# smiDE scale factors
totalcounts <- Matrix::rowSums(counts)
totalcounts_scalefactors <- mean(totalcounts) / totalcounts
names(totalcounts_scalefactors) <- rownames(counts)

# Cell type subset
inds <- (metadata[[celltype_col]] == celltype_oi)
cat("Cells of type '", celltype_oi, "':", sum(inds), "/", length(inds), "\n")
cat("\nDE variable distribution (all cells):\n")
print(table(metadata[[de_var]]))
cat("\nDE variable distribution (", celltype_oi, "):\n")
print(table(metadata[inds, de_var]))

## Celltype-Specific Binning (Optional)

When `bin_within_celltype = TRUE`, bins a continuous predictor column using **only the
distribution of `celltype_oi` cells** to define quantile breakpoints. This ensures
balanced bin sizes within the cell type being tested, maximizing statistical power.

The binned column is written to `metadata[[de_var]]` for use in the smiDE formula.
Non-`celltype_oi` cells also receive a bin assignment (using the same breakpoints),
but they are filtered out by smiDE anyway.

Set `bin_within_celltype = FALSE` if `de_var` already exists as a categorical column
(e.g., a global binning from the Predictors notebook).

In [ ]:
if (bin_within_celltype) {
    x_all <- metadata[[bin_source_col]]
    if (is.null(x_all)) stop("bin_source_col '", bin_source_col, "' not found in metadata.")

    # Compute quantile breakpoints from celltype_oi cells only
    x_ct   <- x_all[inds]
    quants <- quantile(x_ct, probs = seq(0, 1, length.out = bin_n_bins + 1), na.rm = TRUE)
    uq     <- unique(quants)

    if (length(uq) < bin_n_bins + 1) {
        warning("Only ", length(uq) - 1, " unique bins instead of ", bin_n_bins,
                " (", celltype_oi, " values have ties at quantile boundaries).")
    }

    # Bin all cells using celltype_oi-derived breakpoints
    x_binned <- cut(x_all, breaks = uq, include.lowest = TRUE)

    # Apply custom labels
    if (!is.null(bin_labels) && length(bin_labels) == length(levels(x_binned))) {
        levels(x_binned) <- bin_labels
    } else if (!is.null(bin_labels) && length(bin_labels) != length(levels(x_binned))) {
        warning("bin_labels length (", length(bin_labels), ") != actual bins (",
                length(levels(x_binned)), "). Using auto labels.")
    }

    metadata[[de_var]] <- x_binned
    seu@meta.data[[de_var]] <- x_binned

    cat("Created '", de_var, "' by binning '", bin_source_col,
        "' within ", celltype_oi, " quantiles.\n\n")
    cat("Distribution (all cells):\n")
    print(table(metadata[[de_var]], useNA = "ifany"))
    cat("\nDistribution (", celltype_oi, " only):\n")
    print(table(metadata[[de_var]][inds], useNA = "ifany"))
} else {
    cat("Using pre-existing '", de_var, "' column.\n")
    if (!(de_var %in% colnames(metadata))) {
        stop("de_var '", de_var, "' not found in metadata. ",
             "Set bin_within_celltype = TRUE or use a column from the Predictors notebook.")
    }
}

In [ ]:
# Construct the model formula (used by both standard and spatial smiDE)
if (use_random_intercept) {
    formula_str <- paste0("~RankNorm(otherct_expr) + offset(log(nCount_RNA)) + ",
                          de_var, " + (1 | ", random_intercept_col, ")")
} else {
    formula_str <- paste0("~RankNorm(otherct_expr) + offset(log(nCount_RNA)) + ", de_var)
}
cat("Formula:", formula_str, "\n")
formula_obj <- as.formula(formula_str)

## Stage 0: Gene Safety Screening (Overlap Ratio)

Identifies genes whose expression is inflated by segmentation errors.
Genes with `ratio > overlap_threshold` are excluded from DE testing.

In [ ]:
# Overlap ratio is shared across runs (same study, same spatial params)
overlapres_path <- file.path(smide_dir, "overlapres.RDS")

if (compute_overlapres) {
    message("Computing overlap ratios ...")
    overlapres <- smiDE::overlap_ratio_metric(
        assay_matrix = Matrix::t(norm),
        metadata     = cbind(xy, metadata),
        cellid_col   = cellid_col,
        cluster_col  = celltype_col,
        sdimx_col    = sdimx_col,
        sdimy_col    = sdimy_col,
        split_neighbors_by_colname = split_neighbors_by_colname,
        radius = radius
    )
    saveRDS(overlapres, overlapres_path)
    message("Done. Saved to: ", overlapres_path)
} else {
    message("Loading cached overlap ratios from: ", overlapres_path)
    overlapres <- readRDS(overlapres_path)
}

# Filter safe genes for cell type of interest
genes_to_analyze <- overlapres[get(celltype_col) == celltype_oi &
                               ratio <= overlap_threshold]$target

cat("Selected", length(genes_to_analyze), "genes out of", ncol(counts),
    "(", round(length(genes_to_analyze) / ncol(counts) * 100, 1), "%)\n")

saveRDS(genes_to_analyze,
        file.path(results_dir, paste0("genes_to_analyze_", run_tag, ".RDS")))

# Peek at lowest-ratio genes
head(overlapres[get(celltype_col) == celltype_oi][order(ratio)], n = 5)

### Pre-DE: Spatial Neighbor Computation

Pre-computes cell adjacency structures for smiDE's neighbor expression correction.
Shared across runs with the same study and spatial parameters.

In [ ]:
pre_de_path <- file.path(smide_dir, "pre_de_obj.RDS")

if (compute_pre_de) {
    message("Computing pre_de spatial neighborhoods ...")
    pre_de_obj <- smiDE::pre_de(
        metadata                   = cbind(xy, metadata),
        cell_type_metadata_colname = celltype_col,
        cellid_colname             = cellid_col,
        split_neighbors_by_colname = split_neighbors_by_colname,
        mm_radius                  = radius,
        sdimx_colname              = sdimx_col,
        sdimy_colname              = sdimy_col,
        verbose = FALSE
    )
    saveRDS(pre_de_obj, pre_de_path)
    message("Done. Saved to: ", pre_de_path)
} else {
    message("Loading cached pre_de from: ", pre_de_path)
    pre_de_obj <- readRDS(pre_de_path)
}

cat("Adjacency table rows:", nrow(pre_de_obj$cell_adjacency_dt), "\n")

## Stage 1: HastyDE (Fast Exploratory Screen)

Runs a fast, non-spatial DE test on all safe genes.
No neighbor expression correction — good for broad screening and gene ranking.

In [ ]:
if (run_hasty) {
    hasty_path <- file.path(cache_dir, "hasty_fastres.RDS")

    if (compute_hasty) {
        # Gene-scaled normalization for hastyDE
        genefactors <- 1 / pmax(0.05, Matrix::colMeans(norm))
        normscaled  <- norm %*% Matrix::Diagonal(x = genefactors)
        colnames(normscaled) <- colnames(norm)

        # Source hastyDE function
        source("https://raw.githubusercontent.com/Nanostring-Biostats/CosMx-Analysis-Scratch-Space/refs/heads/Main/_code/PatchDE/DEutils.R")

        df_hasty <- as.data.frame(metadata[[de_var]][inds])
        colnames(df_hasty) <- de_var

        message("Running hastyDE on ", length(genes_to_analyze), " genes ...")
        fastres <- hastyDE(y = normscaled[inds, genes_to_analyze], df = df_hasty)
        saveRDS(fastres, hasty_path)
        message("Done. Saved to: ", hasty_path)
    } else {
        message("Loading cached hastyDE from: ", hasty_path)
        fastres <- readRDS(hasty_path)
    }

    cat("HastyDE comparisons:\n")
    print(colnames(fastres$effect))
} else {
    cat("Skipping hastyDE (run_hasty = FALSE)\n")
}

In [ ]:
if (run_hasty) {
    # List available comparisons
    hasty_comparisons <- colnames(fastres$effect)
    cat("Available comparisons:\n")
    for (i in seq_along(hasty_comparisons)) cat(" ", i, ":", hasty_comparisons[i], "\n")

    # Extract results for the first comparison (change index to explore others)
    hasty_comp_idx  <- 1
    hasty_comp_name <- hasty_comparisons[hasty_comp_idx]

    hasty_df <- data.frame(
        gene   = rownames(fastres$effect),
        effect = fastres$effect[, hasty_comp_name],
        p_raw  = fastres$p[, hasty_comp_name],
        stringsAsFactors = FALSE
    )
    rownames(hasty_df) <- hasty_df$gene
    hasty_df$fdr <- p.adjust(hasty_df$p_raw, method = "BH")

    n_sig <- sum(hasty_df$fdr < fdr_cut, na.rm = TRUE)
    cat("\nComparison:", hasty_comp_name, "\n")
    cat("Significant genes (FDR <", fdr_cut, "):", n_sig, "\n")

    write.csv(hasty_df,
              file.path(results_dir, paste0("hasty_results_", run_tag, ".csv")),
              row.names = FALSE)
}

In [ ]:
if (run_hasty) {
    volc_hasty <- plot_volcano(
        df = hasty_df, x_col = "effect", y_col = "fdr",
        x_cut = 0.2, p_cut = fdr_cut,
        genes_of_interest = genes_of_interest,
        y_trans = "-log10"
    )
    print(volc_hasty)

    ggsave(file.path(figures_dir, paste0("volcano_hasty_", run_tag, ".pdf")),
           volc_hasty, width = 10, height = 8)
}

## Stage 2: Standard smiDE (Full Model, No Spatial)

Negative binomial model with neighbor expression correction (`RankNorm(otherct_expr)`).
Runs on all genes that passed the overlap ratio filter.

In [ ]:
if (run_standard) {
    standard_path <- file.path(cache_dir, "standard_de_obj.RDS")

    if (compute_standard) {
        message("Running standard smiDE on ", length(genes_to_analyze), " genes ...")
        de_obj <- smiDE::smi_de(
            assay_matrix    = Matrix::t(counts),
            metadata        = metadata[inds, ],
            formula         = formula_obj,
            pre_de_obj      = pre_de_obj,
            cellid_colname  = cellid_col,
            neighbor_expr_cell_type_metadata_colname = celltype_col,
            neighbor_expr_overlap_weight_colname      = NULL,
            neighbor_expr_overlap_agg                 = "sum",
            neighbor_expr_totalcount_normalize        = TRUE,
            neighbor_expr_totalcount_scalefactor      = totalcounts_scalefactors,
            family  = family,
            targets = genes_to_analyze
        )
        saveRDS(de_obj, standard_path)
        message("Done. Saved to: ", standard_path)
    } else {
        message("Loading cached standard smiDE from: ", standard_path)
        de_obj <- readRDS(standard_path)
    }
    cat("Standard smiDE complete.\n")
} else {
    cat("Skipping standard smiDE (run_standard = FALSE)\n")
}

In [ ]:
if (run_standard) {
    std_results <- extract_smide_results(de_obj, de_var)

    cat("Available contrasts:\n")
    print(unique(std_results$contrast))

    write.csv(std_results,
              file.path(results_dir, paste0("standard_results_pw_", run_tag, ".csv")),
              row.names = FALSE)
    saveRDS(std_results, file.path(cache_dir, "standard_results_extracted.RDS"))

    # smiDE built-in volcano objects
    vlist_standard <- smiDE::volcano(de_obj, comparison = "pairwise",
                                      variable = de_var, interactive = FALSE)
}

In [ ]:
if (run_standard) {
    contrasts_available <- unique(std_results$contrast)

    # Change contrast_idx to explore other contrasts
    contrast_idx <- 1
    comp_name    <- contrasts_available[contrast_idx]
    cat("Analyzing contrast:", comp_name, "\n")

    std_df <- std_results[std_results$contrast == comp_name, ]
    rownames(std_df) <- std_df$gene

    # Top up-regulated
    top_up <- std_df[std_df$fdr < fdr_cut & std_df$log2FC > 0, ]
    top_up <- top_up[order(top_up$log2FC, decreasing = TRUE), ]
    cat("\nTop up-regulated:\n")
    print(head(top_up, top_n_viz))

    # Top down-regulated
    top_down <- std_df[std_df$fdr < fdr_cut & std_df$log2FC < 0, ]
    top_down <- top_down[order(top_down$log2FC), ]
    cat("\nTop down-regulated:\n")
    print(head(top_down, top_n_viz))
}

In [ ]:
if (run_standard) {
    volc_standard <- plot_volcano(
        df = std_df, x_col = "log2FC", y_col = "fdr",
        x_cut = x_cut, p_cut = fdr_cut,
        genes_of_interest = genes_of_interest,
        y_trans = "-log10"
    )
    print(volc_standard)

    ggsave(file.path(figures_dir,
                     paste0("volcano_standard_", gsub("[/ ]", "_", comp_name),
                            "_", run_tag, ".pdf")),
           volc_standard, width = 10, height = 8)
}

## Stage 3: Spatial smiDE (GP Random Effect)

Runs smiDE with a spatial Gaussian Process on the top N genes only.

**Mac Compatibility Issues:**

- **GP_Matern** (via `glmmTMB`): known to fail with `emmeans::ref_grid` error
  ("unable to reconstruct the data"). This is an interaction between `smiDE`,
  `emmeans`, and `glmmTMB`. Try updating all three packages.
- **GP_INLA**: requires INLA installed from `https://inla.r-inla-download.org/R/stable`.
  Can fail on macOS ARM (Apple Silicon).
- **Recommendation**: run standard smiDE (Stage 2) as primary analysis on Mac.
  For spatial DE, use a Linux HPC or cloud environment.
- The spatial call below is wrapped in `tryCatch` so a failure won't crash the notebook.

In [ ]:
if (run_spatial) {
    if (spatial_source == "hasty" && run_hasty) {
        topgenes <- select_top_genes_for_spatial(
            hasty_df, effect_col = "effect", p_col = "p_raw",
            top_n = spatial_top_n, p_quantile = 0.01
        )
    } else if (spatial_source == "standard" && run_standard) {
        topgenes <- select_top_genes_for_spatial(
            std_df, effect_col = "log2FC", p_col = "p_raw",
            top_n = spatial_top_n, p_quantile = 0.01
        )
    } else {
        stop("spatial_source='", spatial_source,
             "' but that stage was not run. Set run_hasty or run_standard = TRUE.")
    }

    cat("Top genes for spatial DE:\n")
    print(topgenes)
    saveRDS(topgenes,
            file.path(results_dir, paste0("topgenes_for_spatial_", run_tag, ".RDS")))
}

In [ ]:
if (run_spatial) {
    spatial_path <- file.path(cache_dir, "spatial_de_obj.RDS")

    if (compute_spatial) {
        # Ensure coordinate columns are in metadata
        metadata[[sdimx_col]] <- xy[, 1]
        metadata[[sdimy_col]] <- xy[, 2]

        # Build spatial_model argument
        if (spatial_method == "GP_INLA") {
            spatial_args <- list(
                name        = "GP_INLA",
                x_coord_col = sdimx_col,
                y_coord_col = sdimy_col,
                quantiles   = c(0.025, 0.5, 0.975),
                num.threads = spatial_num_threads
            )
        } else if (spatial_method == "GP_Matern") {
            spatial_args <- list(
                name                  = "GP_Matern",
                k                     = spatial_k,
                x_coord_col           = sdimx_col,
                y_coord_col           = sdimy_col,
                spatial_random_effect = as.formula(
                    paste0("~Matern(1 | ", sdimx_col, "_cluster + ",
                           sdimy_col, "_cluster)")
                )
            )
        } else {
            stop("Unknown spatial_method: ", spatial_method)
        }

        message("Running spatial smiDE (", spatial_method, ") on ",
                length(topgenes), " genes ...")

        spatial_de_obj <- tryCatch({
            smiDE::smi_de(
                assay_matrix    = Matrix::t(counts),
                metadata        = metadata[inds, ],
                formula         = formula_obj,
                pre_de_obj      = pre_de_obj,
                cellid_colname  = cellid_col,
                neighbor_expr_cell_type_metadata_colname = celltype_col,
                neighbor_expr_overlap_weight_colname      = NULL,
                neighbor_expr_overlap_agg                 = "sum",
                neighbor_expr_totalcount_normalize        = TRUE,
                neighbor_expr_totalcount_scalefactor      = totalcounts_scalefactors,
                family        = family,
                targets       = topgenes,
                spatial_model = spatial_args
            )
        }, error = function(e) {
            warning("Spatial smiDE FAILED: ", conditionMessage(e),
                    "\nFalling back to standard results for these genes.")
            NULL
        })

        if (!is.null(spatial_de_obj)) {
            saveRDS(spatial_de_obj, spatial_path)
            message("Done. Saved to: ", spatial_path)
        } else {
            cat("Spatial smiDE failed. See warning above.\n")
        }
    } else {
        message("Loading cached spatial smiDE from: ", spatial_path)
        spatial_de_obj <- readRDS(spatial_path)
    }
}

In [ ]:
if (run_spatial && exists("spatial_de_obj") && !is.null(spatial_de_obj)) {
    spatial_results <- extract_smide_results(spatial_de_obj, de_var)
    write.csv(spatial_results,
              file.path(results_dir, paste0("spatial_results_", run_tag, ".csv")),
              row.names = FALSE)
    cat("Spatial results saved.\n")
    print(head(spatial_results[order(spatial_results$fdr), ], 10))
} else if (run_spatial) {
    cat("No spatial results available (model failed or not computed).\n")
}

## Visualizations

Plots below use the standard smiDE results (Stage 2) unless otherwise noted.
Plots are saved individually and also compiled into a PDF report.

In [ ]:
# Create cell-type subset for visualization
seu_subset <- subset(seu, cells = colnames(seu)[inds])
cat("Visualization subset:", ncol(seu_subset), "cells\n")

In [ ]:
if (run_standard) {
    sig_genes <- std_df$gene[std_df$fdr < fdr_cut]

    # All genes heatmap
    hm_all <- plot_mean_expression_heatmap(
        seu_subset, bin_col = de_var, genes = NULL, bin_order = bin_order,
        title = paste("All Genes:", celltype_oi, "by", de_var)
    )

    # Significant genes heatmap
    if (length(sig_genes) > 1) {
        hm_sig <- plot_mean_expression_heatmap(
            seu_subset, bin_col = de_var, genes = sig_genes, bin_order = bin_order,
            title = paste("Sig Genes (FDR <", fdr_cut, "):", celltype_oi, "by", de_var)
        )
    }
}

In [ ]:
if (run_standard) {
    # Pick top genes by absolute log2FC (significant only)
    viz_genes <- std_df[std_df$fdr < fdr_cut, ]
    viz_genes <- viz_genes[order(abs(viz_genes$log2FC), decreasing = TRUE), ]
    viz_genes <- head(viz_genes$gene, top_n_viz)

    if (length(viz_genes) > 0) {
        # Dot plot
        dotplot <- generate_spatial_dotplot(
            seu_subset, genes = viz_genes, bin_col = de_var,
            bin_order = bin_order, scale = TRUE
        )
        print(dotplot)

        # Bar plots
        bar_plots <- get_gene_bin_plot_list(
            seu_obj = seu_subset, genes = viz_genes, bin_col = de_var,
            bin_order = bin_order
        )
        print(bar_plots)
    } else {
        cat("No significant genes to visualize.\n")
    }
}

In [ ]:
if (run_standard && exists("viz_genes") && length(viz_genes) > 0) {
    spatial_expr_plots <- lapply(head(viz_genes, 4), function(g) {
        plot_gene_spatial_expression(
            seu_subset, gene = g, x_col = sdimx_col, y_col = sdimy_col
        )
    })
    print(wrap_plots(spatial_expr_plots, ncol = 2))
}

## Compiled PDF Report

In [ ]:
# Collect all plots into a named list
report_plots <- list()

if (run_hasty && exists("volc_hasty")) {
    report_plots[["Volcano: HastyDE"]] <- volc_hasty
}

if (run_standard) {
    if (exists("volc_standard"))  report_plots[["Volcano: Standard smiDE"]] <- volc_standard
    if (exists("hm_all"))         report_plots[["Heatmap: All Genes"]]      <- hm_all
    if (exists("hm_sig"))         report_plots[["Heatmap: Sig Genes"]]      <- hm_sig
    if (exists("dotplot"))        report_plots[["DotPlot: Top Genes"]]      <- dotplot
    if (exists("bar_plots"))      report_plots[["BarPlots: Top Genes"]]     <- bar_plots
    if (exists("spatial_expr_plots")) {
        for (i in seq_along(spatial_expr_plots)) {
            report_plots[[paste0("SpatialExpr: ", viz_genes[i])]] <- spatial_expr_plots[[i]]
        }
    }
    if (exists("vlist_standard")) report_plots[["smiDE Volcano (built-in)"]] <- vlist_standard
}

report_path <- file.path(run_out_dir, paste0("report_", run_tag, ".pdf"))
generate_smide_report(
    plots = report_plots, run_tag = run_tag,
    config = config, output_file = report_path
)

## P-value Correction for Selection Bias

For genes omitted at the overlap ratio step, their p-values are replaced
with a uniform null distribution before BH correction. This ensures the FDR
is calculated over the full gene set, not just the tested subset.

See NanoString vignette for rationale.

In [ ]:
if (run_standard) {
    all_genes    <- colnames(counts)
    tested_genes <- genes_to_analyze
    omitted_genes <- setdiff(all_genes, tested_genes)

    corrected_results <- do.call(rbind, lapply(
        unique(std_results$contrast), function(ct) {
            sub <- std_results[std_results$contrast == ct, ]

            # Uniform null p-values for omitted genes
            set.seed(42)
            null_df <- data.frame(
                gene     = omitted_genes,
                contrast = ct,
                FC       = 1,
                log2FC   = 0,
                p_raw    = runif(length(omitted_genes)),
                fdr      = NA,
                stringsAsFactors = FALSE
            )

            combined <- rbind(sub, null_df)
            combined$fdr_corrected <- p.adjust(combined$p_raw, method = "BH")
            combined
        }
    ))

    write.csv(corrected_results,
              file.path(results_dir,
                        paste0("standard_results_corrected_", run_tag, ".csv")),
              row.names = FALSE)
    cat("Corrected results saved with", length(all_genes), "genes per contrast.\n")
}

## Session Info

In [ ]:
sessionInfo()